# Reward Model Debug

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

import sumo_rl_ego as sre
from human_feedback_rl.common.trajectory_generators import TrajectoryGeneratorFromAgent

from loadings import load_reward_ensemble
from plot_functions import plot_scatter, plot_reward_curves

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

In [ ]:
CHECKPOINT_DIR = Path("/Users/andreazhang/Documents/fis/sumo-human-feedback-rl/outputs/meeting/seg-length-study/seg1_comps100_seed3/checkpoint_0050")

import sumo_gym_ego.core.simulation as _sim_mod
import traci as _traci_mod
_sim_mod.load_traci = lambda use_gui: _traci_mod

try:
    env.close()
except Exception:
    pass

env = sre.make_vec_env(
    "HighwayEgo-v0",
    n_envs=1,
    base_seed=0,
    ego="continuous",
    reward="fast"
)

reward_model = load_reward_ensemble(
    CHECKPOINT_DIR / "reward_model.pt",
    env.observation_space,
    env.action_space,
)

# Passa env a load() per fare l'override di n_envs (il modello era salvato con n_envs=4)
agent = PPO.load(CHECKPOINT_DIR / "agent.zip", env=env, device="cpu")

trajectory_generator = TrajectoryGeneratorFromAgent(
    agent=agent,
    reward_model=reward_model,
    venv=env,
)

In [ ]:
N_STEPS = 1000
trajectories, log_metrics = trajectory_generator.sample(N_STEPS)

In [ ]:
import numpy as np

# next_status: 7-dim one-hot [arrived, collided, off_road, timeout, running, teleported, removed_unknown]
STATUS_ARRIVED  = 0
STATUS_COLLIDED = 1
STATUS_OFFROAD  = 2
STATUS_TIMEOUT  = 3

terminal_statuses = [np.argmax(traj[-1].next_status) for traj in trajectories]

n_episodes = len(trajectories)
n_arrived  = sum(s == STATUS_ARRIVED  for s in terminal_statuses)
n_collided = sum(s == STATUS_COLLIDED for s in terminal_statuses)
n_offroad  = sum(s == STATUS_OFFROAD  for s in terminal_statuses)
n_timeout  = sum(s == STATUS_TIMEOUT  for s in terminal_statuses)

print(f"Episodi totali:        {n_episodes}")
print(f"Lunghezza media:       {log_metrics.mean_length:.1f}")
print(f"Reward media:          {log_metrics.mean_true_reward:.3f}")
print(f"Reward predetta media: {log_metrics.mean_model_reward:.3f}")
print()
print(f"Terminazioni:")
print(f"  Arrived:  {n_arrived:3d}  ({100 * n_arrived  / n_episodes:.1f}%)")
print(f"  Collided: {n_collided:3d}  ({100 * n_collided / n_episodes:.1f}%)")
print(f"  Off-road: {n_offroad:3d}  ({100 * n_offroad  / n_episodes:.1f}%)")
print(f"  Timeout:  {n_timeout:3d}  ({100 * n_timeout  / n_episodes:.1f}%)")

In [ ]:
for seg_len in [1, 5, 20, None]:
    plot_scatter(trajectories, reward_model, segment_length=seg_len, normalize=False)

for seg_len in [1, 5, 20, None]:
    plot_scatter(trajectories, reward_model, segment_length=seg_len, normalize=True, robust=False)

for seg_len in [1, 5, 20, None]:
    plot_scatter(trajectories, reward_model, segment_length=seg_len, normalize=True, robust=True)

In [ ]:
plot_reward_curves(trajectories, reward_model, normalize=True, robust=True)